### 1. Configuration


In [1]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import numpy as np
import wave
from scipy.signal import firwin, fftconvolve, butter, sosfiltfilt, resample_poly
from scipy.ndimage import maximum_filter1d
from fractions import Fraction
from functools import lru_cache
import torch
from torchaudio.functional import melscale_fbanks, lfilter
import pandas as pd


DATA = Path("/media/ensound/Ventoy/apnea_data")
names = ("pressure", "thermistor", "effort_tho", "effort_abd")
labels = ("CanFlow", "TFlow", "THO", "ABD")


In [2]:
def load_signals(patient):
    parts= []
    for path in sorted((DATA / "signals" / patient).glob("*.npz")):
        if path.name.endswith(".correlation.npz"):
            continue
        with np.load(path) as data:
            part = np.column_stack([data[name] for name in names]).astype(np.float32)
        parts.append(part)
    return np.concatenate(parts)

def load_masks(patient):
    root = ET.parse(DATA / "annotations" / f"{patient}.rml").getroot()
    channels = {e.get("EdfSignal"): e.find("{*}Label").text for e in root.iter() if e.tag.rsplit("}", 1)[-1] == "Channel" and e.find("{*}Label") is not None}
    duration = float(root.find(".//{*}Session/{*}Duration").text)
    valid = np.ones((int(np.ceil(duration * 100)), len(labels)), dtype=bool)
    for event in root.iter():
        if event.get("Type") != "ChannelFail":
            continue
        start = float(event.get("Start"))
        end = start + float(event.get("Duration"))
        a, b = int(np.ceil(start * 100)), int(np.ceil(end * 100))
        for flag in event:
            label = channels.get(flag.get("EdfSignal"))
            if label in labels:
                valid[a:b, labels.index(label)] = False
    valid = (maximum_filter1d((~valid).astype(np.uint8), 161, axis=0, mode="constant", cval=1) == 0)[::10]
    return valid

def load_audio(patient):
    folder = DATA / "audio" / patient
    metadata = json.loads((folder / "metadata.json").read_text())
    parts = []
    segments = metadata["segments"]
    for i, segment in enumerate(segments):
        count = segment["end_valid"] if i == len(segments) - 1 else segment["count"]
        with wave.open(str(folder / segment["file"])) as reader:
            pcm = np.frombuffer(reader.readframes(count), dtype="<i2")
        parts.append(pcm)
    audio = np.concatenate(parts).astype(np.float32) / 32768
    end = max(
        (sum(s["count"] for s in segments[:i]) + segment["end_valid"]
        for i, segment in enumerate(segments) if segment["end_valid"] > 0),
        default=0
    )
    return audio[:end]

@lru_cache(maxsize=None)
def load_events(patient):
    root = ET.parse(DATA / "annotations" / f"{patient}.rml").getroot()
    events = []
    for event in root.iter():
        kind = event.get("Type", "")
        if event.get("Family") == "Respiratory" and ("Apnea" in kind or "Hypopnea" in kind):
            start = float(event.get("Start"))
            events.append([start, start + float(event.get("Duration"))])
    merged = []
    for a, b in sorted(events):
        if merged and a <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    return merged


In [3]:
def correct_pressure_polarity(signals, valid):
    sos = butter(3, [0.1, 0.8], fs=10, btype="bandpass", output="sos")
    filtered, support = np.zeros_like(signals[:, [0, 3]]), np.zeros_like(valid[:, [0, 3]])
    for c, source in enumerate([0, 3]):
        edges = np.diff(np.r_[False, valid[:, source], False].astype(int))
        for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
            if b - a >= 800:
                filtered[a:b, c] = sosfiltfilt(sos, signals[a:b, source])
                support[a + 100:b - 100, c] = True
    derivative = np.gradient(filtered[:, 1])
    correlations = []
    for a in range(0, len(signals) - 599, 600):
        mask = support[a:a + 600].all(axis=1)
        if mask.sum() >= 540:
            x, z = filtered[a:a + 600, 0][mask], derivative[a:a + 600][mask]
            x, z = x - x.mean(), z - z.mean()
            correlations.append(x @ z / max(np.linalg.norm(x) * np.linalg.norm(z), 1e-12))
    assert correlations, "No valid windows for pressure polarity"
    score = float(np.median(correlations))
    sign = -1 if score >= 0 else 1
    signals[:, 0] *= sign
    return signals

In [4]:
OUT = Path("./dataset")
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (2026, 2027, 2028)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#Parameters
WINDOW, CONTEXT = 60, 600 #s
STFT_WINDOW, STFT_HOP = 40, 10 #ms
MEL_BANDS = 128 

In [5]:
def align_audio(audio, patient):
    fit = json.loads((DATA / "alignment" / patient / f"{patient}.json").read_text())
    ratio = Fraction(fit["a"]).limit_denominator(100000)
    up, down, shift = ratio.numerator, ratio.denominator, round(fit["b"] * 48000)
    if up != down:
        kernel = firwin(20 * max(up, down) + 1, 1 / max(up, down), window=("kaiser", 5)).astype(np.float32)
        audio = resample_poly(audio, up, down, window=kernel)
    if shift > 0:
        audio = np.pad(audio, (shift, 0))
    elif shift < 0:
        audio = audio[-shift:]
    return audio, shift

def normalize_targets(signals, valid):
    window = round(CONTEXT * 10)
    values = pd.DataFrame(signals).where(valid)
    rolling = values.rolling(window, center=True, min_periods=window // 2)
    median = rolling.median().to_numpy()
    scale = (rolling.quantile(0.95) - rolling.quantile(0.05)).to_numpy()
    mask = valid & np.isfinite(median) & np.isfinite(scale) & (scale > 0)
    target = np.zeros_like(signals, dtype=np.float32)
    np.divide(signals - median, scale, out=target, where=mask)
    mask &= np.abs(target) < 10
    return target, mask

def prepare_signals(patient):
    signals, valid = load_signals(patient), load_masks(patient)
    fir = firwin(161, 3.5, fs=100, window=("kaiser", 8))
    signals = fftconvolve(signals, fir[:, None], mode="same", axes=0)[::10].astype(np.float32)
    signals = correct_pressure_polarity(signals, valid)
    signals, valid = normalize_targets(signals, valid)
    return signals, valid

In [6]:
N_FFT = round(STFT_WINDOW * 48000 / 1000)
HOP = round(STFT_HOP * 48000 / 1000)

hann = torch.hann_window(N_FFT, device=DEVICE)
mel_filters = melscale_fbanks(N_FFT // 2 + 1, 0, 24000, MEL_BANDS, 48000, norm="slaney", mel_scale="slaney").T.to(DEVICE)

@torch.inference_mode()
def make_pcen_image(audio):
    audio = torch.as_tensor(audio, dtype=torch.float32, device=DEVICE)
    spectrum = torch.stft(audio, n_fft=N_FFT, hop_length=HOP, window=hann, 
                          center=True, pad_mode="reflect", return_complex=True).abs()[:, :-1]
    energy = (mel_filters @ spectrum).clamp_min(1e-5)
    a = energy.new_tensor([1, -0.975])
    b = energy.new_tensor([0.025, 0])
    smooth = lfilter(energy, a, b, clamp=False)
    decay = (0.975 ** torch.arange(1, energy.shape[-1] + 1, dtype=torch.float64, device=DEVICE)).float()
    smooth += energy[:, :1] * decay
    image = torch.sqrt(energy / (1e-6 + smooth).pow(0.98) + 2) - np.sqrt(2)
    return image.cpu().numpy()

In [7]:
def label_apnea(window, events):
    overlap = sum(max(0, min(window[1], b) - max(window[0], a)) for a, b in events)
    return 1 if overlap >= 10 else 0 if overlap == 0 else -1

def select_windows(audio, signals, valid, valid_start):
    margin = (CONTEXT - WINDOW) / 2
    return [(start, start+WINDOW)
        for start in range(0, len(audio) // 48000 - WINDOW + 1, WINDOW)
        if start >= max(valid_start / 48000, margin, 0)]

In [8]:
patients = []
for folder in (DATA / "audio").iterdir():
    patient = folder.name
    fit = json.loads((DATA / "alignment" / patient / f"{patient}.json").read_text())
    if fit.get("a") is None or fit.get("b") is None:
        continue
    patients.append(folder.name)

patients = sorted(patients)
nvt = round(len(patients) * 0.15)
splits = {}
for seed in SEEDS:
    order = np.random.default_rng(seed).permutation(patients).tolist()
    splits[seed] = dict(
        train=order[2*nvt:],
        validation=order[:nvt],
        test=order[nvt:2*nvt],
    )

(OUT / "splits.json").write_text(json.dumps(splits, indent=2))

12005

In [ ]:
done_folder = OUT / "done"
done_folder.mkdir(parents=True, exist_ok=True)
for patient in patients:
    images_folder = OUT / "images" / patient
    signals_folder = OUT / "signals" / patient
    done = done_folder / f"{patient}.done"
    if done.exists():
        continue
    images_folder.mkdir(parents=True, exist_ok=True)
    signals_folder.mkdir(parents=True, exist_ok=True)

    signals, valid = prepare_signals(patient)
    audio = load_audio(patient)
    audio, shift = align_audio(audio, patient)
    windows = select_windows(audio, signals, valid, max(0, shift))
    events = load_events(patient)
    for window in windows:
        image = make_pcen_image(audio[window[0]*48000: window[1]*48000])
        label = label_apnea(window, events)
        np.save(images_folder / f"{window[0]:06d}.npy", image.astype(np.float16))
        np.savez_compressed(signals_folder / f"{window[0]:06d}.npz", target=signals[window[0]*10:window[1]*10], valid=valid[window[0]*10:window[1]*10], label=label)
    done.touch()
    print(f"Done patient {patient}")
    del audio, signals, valid

Done patient 1000
